# SQL practice 1: query fundamentals

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-01-query-fundamentals.ipynb)

14 exercises on filtering, pattern matching, sorting, paging and projection. Each exercise names the [reference entries](sql-reference-01-query-fundamentals.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 1](sql-reference-01-query-fundamentals.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

11 easy, 3 medium, 0 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Paid orders over a threshold](#exercise-1-paid-orders-over-a-threshold) | Easy | [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators) |
| 2 | [Orders in a date range](#exercise-2-orders-in-a-date-range) | Easy | [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators) |
| 3 | [Case-insensitive name search](#exercise-3-case-insensitive-name-search) | Easy | [Pattern matching: `LIKE`, `ILIKE`, `RLIKE`](sql-reference-01-query-fundamentals.ipynb#pattern-matching-like-ilike-rlike) |
| 4 | [SKU codes by pattern](#exercise-4-sku-codes-by-pattern) | Medium | [Pattern matching: `LIKE`, `ILIKE`, `RLIKE`](sql-reference-01-query-fundamentals.ipynb#pattern-matching-like-ilike-rlike) |
| 5 | [Top three orders with a stable tie-break](#exercise-5-top-three-orders-with-a-stable-tie-break) | Easy | [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset) |
| 6 | [Second page of results](#exercise-6-second-page-of-results) | Easy | [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset) |
| 7 | [NULLs last in a sort](#exercise-7-nulls-last-in-a-sort) | Easy | [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset) |
| 8 | [Distinct status pairs](#exercise-8-distinct-status-pairs) | Easy | [`DISTINCT`](sql-reference-01-query-fundamentals.ipynb#distinct) |
| 9 | [Largest order lines](#exercise-9-largest-order-lines) | Easy | [`SELECT` list, aliases and `*`](sql-reference-01-query-fundamentals.ipynb#select-list-aliases-and-) |
| 10 | [Precedence of AND and OR](#exercise-10-precedence-of-and-and-or) | Medium | [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators) |
| 11 | [Hires in selected departments](#exercise-11-hires-in-selected-departments) | Easy | [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators) |
| 12 | [Sort by a computed key](#exercise-12-sort-by-a-computed-key) | Easy | [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset) |
| 13 | [All columns but one](#exercise-13-all-columns-but-one) | Easy | [`SELECT` list, aliases and `*`](sql-reference-01-query-fundamentals.ipynb#select-list-aliases-and-) |
| 14 | [Rewrite with pipe syntax](#exercise-14-rewrite-with-pipe-syntax) | Medium | [SQL pipe syntax (Spark 4.0)](sql-reference-01-query-fundamentals.ipynb#sql-pipe-syntax-spark-40) |

## Exercise 1: Paid orders over a threshold

**Difficulty:** Easy · about 3 min · **Practises:** [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators)

*The most common query there is: filter, project, sort.*

Return `order_id`, `customer_id` and `amount` of `paid` orders with an amount of at least 100, ordered by `amount` descending.

**Expected output**

| order_id | customer_id | amount |
| --- | --- | --- |
| 7 | 3 | 310.00 |
| 4 | 2 | 260.00 |
| 1 | 1 | 120.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, customer_id, amount
    FROM orders
    WHERE status = 'paid' AND amount >= 100
    ORDER BY amount DESC, order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Orders in a date range

**Difficulty:** Easy · about 5 min · **Practises:** [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators)

*Daily reports must include the whole last day, including late-evening orders.*

Return `order_id` and `order_ts` for orders placed from 2026-01-05 through 2026-01-07 **inclusive** (whole days), ordered by `order_ts`.

**Expected output**

| order_id | order_ts |
| --- | --- |
| 1 | 2026-01-05 09:15:00 |
| 2 | 2026-01-05 10:05:00 |
| 3 | 2026-01-05 11:40:00 |
| 4 | 2026-01-06 15:20:00 |
| 5 | 2026-01-07 08:05:00 |
| 6 | 2026-01-07 17:45:00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, order_ts
    FROM orders
    WHERE order_ts >= TIMESTAMP'2026-01-05 00:00:00'
      AND order_ts <  TIMESTAMP'2026-01-08 00:00:00'
    ORDER BY order_ts
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Case-insensitive name search

**Difficulty:** Easy · about 3 min · **Practises:** [Pattern matching: `LIKE`, `ILIKE`, `RLIKE`](sql-reference-01-query-fundamentals.ipynb#pattern-matching-like-ilike-rlike)

*Search boxes match regardless of case.*

Return the `name` of customers whose name contains `retail` or `foods` in any letter case, ordered by name.

**Expected output**

| name |
| --- |
| Bright Retail |
| Foxtrot Foods |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT name
    FROM customers
    WHERE name ILIKE '%retail%' OR name ILIKE '%foods%'
    ORDER BY name
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: SKU codes by pattern

**Difficulty:** Medium · about 5 min · **Practises:** [Pattern matching: `LIKE`, `ILIKE`, `RLIKE`](sql-reference-01-query-fundamentals.ipynb#pattern-matching-like-ilike-rlike)

*Product codes encode families; regular expressions pick them out.*

Return the `sku` and `category` of products whose SKU number (after `SKU-`) is between 4005 and 4009 inclusive, using a regular expression, ordered by `sku`.

**Expected output**

| sku | category |
| --- | --- |
| SKU-4005 | Pantry |
| SKU-4006 | Outdoor |
| SKU-4007 | Kitchen |
| SKU-4008 | Outdoor |
| SKU-4009 | Pantry |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT sku, category
    FROM products
    WHERE sku RLIKE '^SKU-400[5-9]$'
    ORDER BY sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Top three orders with a stable tie-break

**Difficulty:** Easy · about 3 min · **Practises:** [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset)

*Leaderboards must not change between runs when values tie.*

Return `order_id` and `amount` of the three largest orders; break ties by the smaller `order_id`.

**Expected output**

| order_id | amount |
| --- | --- |
| 9 | 1180.00 |
| 7 | 310.00 |
| 4 | 260.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, amount
    FROM orders
    ORDER BY amount DESC, order_id
    LIMIT 3
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Second page of results

**Difficulty:** Easy · about 5 min · **Practises:** [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset)

*APIs page through results in a fixed order.*

Return `order_id` and `order_ts` for the second page of orders, 5 per page, ordered by `order_ts` then `order_id`.

**Expected output**

| order_id | order_ts |
| --- | --- |
| 6 | 2026-01-07 17:45:00 |
| 7 | 2026-01-09 12:10:00 |
| 8 | 2026-01-10 19:30:00 |
| 9 | 2026-01-12 08:55:00 |
| 10 | 2026-01-12 13:00:00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, order_ts
    FROM orders
    ORDER BY order_ts, order_id
    LIMIT 5 OFFSET 5
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: NULLs last in a sort

**Difficulty:** Easy · about 3 min · **Practises:** [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset)

*Reports put unknown values at the end, whatever the engine default is.*

Return `customer_id`, `name` and `region` for all customers ordered by `region` ascending with customers that have no region last, then by `customer_id`.

**Expected output**

| customer_id | name | region |
| --- | --- | --- |
| 7 | Gamma Sports | East |
| 8 | Hotel Homeware | East |
| 1 | Acme Corp | North |
| 2 | Bright Retail | North |
| 3 | Cyan Traders | South |
| 5 | Echo Supplies | West |
| 6 | Foxtrot Foods | West |
| 4 | Delta Grocers | null |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, name, region
    FROM customers
    ORDER BY region ASC NULLS LAST, customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Distinct status pairs

**Difficulty:** Easy · about 3 min · **Practises:** [`DISTINCT`](sql-reference-01-query-fundamentals.ipynb#distinct)

*Which customers have orders in which states?*

Return the distinct `(customer_id, status)` pairs for customers 1 to 3, ordered by both columns.

**Expected output**

| customer_id | status |
| --- | --- |
| 1 | paid |
| 1 | returned |
| 1 | shipped |
| 2 | paid |
| 2 | placed |
| 3 | paid |
| 3 | shipped |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT DISTINCT customer_id, status
    FROM orders
    WHERE customer_id BETWEEN 1 AND 3
    ORDER BY customer_id, status
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Largest order lines

**Difficulty:** Easy · about 5 min · **Practises:** [`SELECT` list, aliases and `*`](sql-reference-01-query-fundamentals.ipynb#select-list-aliases-and-)

*Line-level revenue drives basket analysis.*

Return `order_id`, `sku` and `line_total` (`qty * unit_price`) for the five largest order lines, largest first, ties by `order_id` then `sku`.

**Expected output**

| order_id | sku | line_total |
| --- | --- | --- |
| 9 | SKU-4011 | 1180.00 |
| 11 | SKU-4013 | 210.00 |
| 7 | SKU-4006 | 189.00 |
| 4 | SKU-4003 | 149.00 |
| 6 | SKU-4003 | 149.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, sku, qty * unit_price AS line_total
    FROM order_items
    ORDER BY line_total DESC, order_id, sku
    LIMIT 5
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Precedence of AND and OR

**Difficulty:** Medium · about 5 min · **Practises:** [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators)

*Mixing AND and OR without parentheses is one of the most common silent bugs in SQL.*

Return `order_id`, `status` and `amount` of orders that are `paid` or `shipped` **and** have an amount over 100, ordered by `order_id`.

**Expected output**

| order_id | status | amount |
| --- | --- | --- |
| 1 | paid | 120.00 |
| 4 | paid | 260.00 |
| 6 | shipped | 149.00 |
| 7 | paid | 310.00 |
| 9 | shipped | 1180.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, status, amount
    FROM orders
    WHERE status IN ('paid', 'shipped') AND amount > 100
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Hires in selected departments

**Difficulty:** Easy · about 3 min · **Practises:** [Comparison and logical operators](sql-reference-01-query-fundamentals.ipynb#comparison-and-logical-operators)

*Membership lists keep filters readable.*

Return `name`, `dept` and `hire_date` of employees in Sales or Finance hired in 2020 or later, ordered by `hire_date`.

**Expected output**

| name | dept | hire_date |
| --- | --- | --- |
| Farah | Sales | 2021-05-17 |
| Gus | Sales | 2023-01-09 |
| Ivan | Finance | 2024-04-01 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT name, dept, hire_date
    FROM employees
    WHERE dept IN ('Sales', 'Finance') AND hire_date >= DATE'2020-01-01'
    ORDER BY hire_date
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Sort by a computed key

**Difficulty:** Easy · about 3 min · **Practises:** [`ORDER BY`, `LIMIT` and `OFFSET`](sql-reference-01-query-fundamentals.ipynb#order-by-limit-and-offset)

*Sort keys can be expressions that do not appear in the output.*

Return customer `name`s ordered by name length, longest first, then alphabetically.

**Expected output**

| name |
| --- |
| Hotel Homeware |
| Bright Retail |
| Delta Grocers |
| Echo Supplies |
| Foxtrot Foods |
| Cyan Traders |
| Gamma Sports |
| Acme Corp |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT name
    FROM customers
    ORDER BY length(name) DESC, name
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: All columns but one

**Difficulty:** Easy · about 3 min · **Practises:** [`SELECT` list, aliases and `*`](sql-reference-01-query-fundamentals.ipynb#select-list-aliases-and-)

*Dropping a wide or sensitive column without listing every other one.*

Return every column of `orders` except `order_ts`, for customer 4, ordered by `order_id`.

**Expected output**

| order_id | customer_id | status | amount |
| --- | --- | --- | --- |
| 8 | 4 | returned | 89.90 |
| 9 | 4 | shipped | 1180.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT * EXCEPT (order_ts)
    FROM orders
    WHERE customer_id = 4
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Rewrite with pipe syntax

**Difficulty:** Medium · about 10 min · **Practises:** [SQL pipe syntax (Spark 4.0)](sql-reference-01-query-fundamentals.ipynb#sql-pipe-syntax-spark-40)

*Pipe syntax reads in execution order, which helps with long transformations.*

Using pipe syntax, return `customer_id`, `orders` (count) and `total` (sum of `amount`) over non-returned orders, keeping customers with a total above 200, ordered by `total` descending.

**Expected output**

| customer_id | orders | total |
| --- | --- | --- |
| 4 | 1 | 1180.00 |
| 3 | 2 | 459.00 |
| 2 | 2 | 315.25 |
| 6 | 2 | 306.40 |
| 1 | 2 | 204.50 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    FROM orders
    |> WHERE status <> 'returned'
    |> AGGREGATE count(*) AS orders, sum(amount) AS total GROUP BY customer_id
    |> WHERE total > 200
    |> ORDER BY total DESC
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 1: query fundamentals](sql-reference-01-query-fundamentals.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).